In [7]:
import pandas as pd
from collections import Counter
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold, TimeSeriesSplit, GroupKFold
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import shap
import optuna
from optuna import Trial
import warnings
warnings.filterwarnings('ignore')

## Dataset Preparation

In [8]:
df = pd.read_json('data/train.json')
df_test = pd.read_json('data/test.json')
df.head()

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


In [9]:
mapping = {'high': 2, 'medium': 1, 'low': 0}

df['interest_level'] = df['interest_level'].map(mapping)

lower_bound = df['price'].quantile(0.01) 
upper_bound = df['price'].quantile(0.99)
df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]

lower_bound = df_test['price'].quantile(0.01) 
upper_bound = df_test['price'].quantile(0.99)
df_test = df_test[(df_test['price'] >= lower_bound) & (df_test['price'] <= upper_bound)]

new_features = list(df['features'])
features_set = []
stop_char = [',', ' ', '"', "'", '*']
for i in range(len(new_features)):
    for j in range(len(new_features[i])):
        for char in stop_char:
            new_features[i][j] = new_features[i][j].replace(char, "")
        features_set.append(new_features[i][j])
df['features'] = new_features

new_features_test = list(df_test['features'])
features_set_test = []
for i in range(len(new_features_test)):
    for j in range(len(new_features_test[i])):
        for char in stop_char:
            new_features_test[i][j] = new_features_test[i][j].replace(char, "")
        features_set_test.append(new_features_test[i][j])
df_test['features'] = new_features_test

feature_count = Counter(features_set)
top_20 = [i[0] for i in feature_count.most_common(20)]

feature_list = ['bathrooms', 'bedrooms', 'interest_level']
for i in range(20):
    name = top_20[i]
    feature_list.append(name)
    df[name] = 0
    df_test[name] = 0
    df.loc[df['features'].apply(lambda x: top_20[i] in x), name] = 1
    df_test.loc[df_test['features'].apply(lambda x: top_20[i] in x), name] = 1
    
df['created'] = pd.to_datetime(df['created'])
df = df.sort_values('created')
df.tail()
print(df.columns)

Index(['bathrooms', 'bedrooms', 'building_id', 'created', 'description',
       'display_address', 'features', 'latitude', 'listing_id', 'longitude',
       'manager_id', 'photos', 'price', 'street_address', 'interest_level',
       'Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman',
       'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War',
       'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom',
       'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding',
       'NewConstruction', 'Terrace'],
      dtype='object')


In [10]:
X = df[feature_list]
y = df['price']

print(df['created'])

111817   2016-04-01 22:12:41
117995   2016-04-01 22:56:00
114617   2016-04-01 22:57:15
117474   2016-04-01 23:26:07
103891   2016-04-02 00:48:13
                 ...        
335      2016-06-29 17:47:34
26349    2016-06-29 17:56:12
622      2016-06-29 18:14:48
34914    2016-06-29 18:30:41
28466    2016-06-29 21:41:47
Name: created, Length: 48379, dtype: datetime64[ns]


### Implementing split methods:

Split data into 2 parts randomly with parameter test_size (ratio from 0 to 1), return training and test samples.

In [11]:
def split_test_train(X, y, test_size, random_state = None):
    
    if random_state is not None:
        np.random.seed(random_state)
        
    n_samples = len(X)
    n_test = int(len(X)*test_size)
    
    indices = np.random.permutation(n_samples)
    
    test_indices = indices[:n_test]
    train_indices = indices[n_test:]
    
    X_train = X.iloc[train_indices]
    X_test = X.iloc[test_indices]
    y_train = y.iloc[train_indices]
    y_test = y.iloc[test_indices]
    
    return X_train, X_test, y_train, y_test    

X_train, X_test, y_train, y_test = split_test_train(X, y, test_size=0.2, random_state=21)
X_test

,bathrooms,bedrooms,interest_level,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
101567,1.0,1,0,1,0,1,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
10287,2.0,3,2,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59586,1.0,2,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
118632,1.0,2,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9824,1.0,1,0,1,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36655,1.0,0,0,1,1,1,1,1,1,1,...,0,1,0,0,1,0,0,0,0,0
12789,1.0,1,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
115706,1.0,2,0,0,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
39499,1.0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Randomly split data into 3 parts with parameters validation_size and test_size, return train, validation and test samples.

In [12]:
def split_data(X, y, test_size, val_size, random_state=None):
    
    if random_state is not None:
        np.random.seed(random_state)
        
    n_samples = len(X)
    
    n_test = int(n_samples * test_size)
    n_val = int(n_samples * val_size)
    n_train = n_samples - n_test - n_val
    
    indices = np.random.permutation(n_samples)
    
    test_indices = indices[:n_test]
    val_indices = indices[n_test:n_test + n_val]
    train_indices = indices[n_test + n_val:]
    
    X_train = X.iloc[train_indices]
    X_val = X.iloc[val_indices]
    X_test = X.iloc[test_indices]
    y_train = y.iloc[train_indices]
    y_val = y.iloc[val_indices]
    y_test = y.iloc[test_indices]
    
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, test_size=0.2, val_size=0.3, random_state=21)

X_train

,bathrooms,bedrooms,interest_level,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
3057,1.0,3,2,0,1,1,1,1,1,1,...,1,0,0,0,0,0,0,0,1,0
2397,1.0,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
31003,1.0,2,0,1,1,1,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0
83242,2.0,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
86463,2.0,2,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51413,1.0,3,1,0,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
120088,1.0,0,0,1,0,1,1,0,1,1,...,1,0,0,0,0,0,0,0,0,0
87477,1.0,2,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
107687,1.0,2,1,1,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Split data into 2 parts with parameter date_split, return train and test samples split by date_split param.

In [13]:
def split_train_test_date(X, y, date_split):
        
    datetime = pd.to_datetime(date_split)
    
    train_indices = df[df['created'] < datetime].index   
    test_indices = df[df['created'] >= datetime].index
    
    X_train = X.loc[train_indices]
    X_test = X.loc[test_indices]
    y_train = y.loc[train_indices]
    y_test = y.loc[test_indices]
    
    return X_train, X_test, y_train, y_test    
    
X_train, X_test, y_train, y_test = split_train_test_date(X, y, date_split='2016-06-29 17:56:12')
X_test

,bathrooms,bedrooms,interest_level,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
26349,1.0,1,0,1,0,1,1,0,1,0,...,0,0,1,0,0,0,0,1,0,0
622,1.0,1,0,1,0,1,1,0,1,0,...,0,0,0,0,0,0,0,1,0,0
34914,1.0,3,1,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
28466,1.0,3,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Split data into 3 parts with parameters validation_date and test_date, return train, validation and test samples split by input params.

In [14]:
def split_by_date(X, y, validation_date, test_date):
        
    test_datetime = pd.to_datetime(test_date)
    val_datetime = pd.to_datetime(validation_date)
    
    train_indices = df[df['created'] < val_datetime].index   
    val_indices = df[df['created'] >= val_datetime]
    val_indices = val_indices[val_indices['created'] < test_datetime].index
    test_indices = df[df['created'] >= test_datetime].index

    X_train = X.loc[train_indices]
    X_test = X.loc[test_indices]
    X_val = X.loc[val_indices]
    y_train = y.loc[train_indices]
    y_test = y.loc[test_indices]
    y_val = y.loc[val_indices]
    
    return X_train, X_test, X_val, y_train, y_test, y_val    

X_train, X_test, X_val, y_train, y_test, y_val = split_by_date(X, y, validation_date='2016-04-02', test_date='2016-06-29')


### Implementint cross-validation methods:

K-Fold, where k is the input parameter, returns a list of train and test indices.

In [15]:
def my_KFold(X, y, k, random_state=None, shuffle=True):
    
    n_samples = len(X)
    n_fold = int(len(X)/k)
    result = []
    
    if shuffle==True:
        if random_state != None:
            np.random.seed(random_state)
        indices = np.random.permutation(n_samples)  
    else:
        indices = np.arange(n_samples)  
    
    for i in range(k):
        if i == k-1:
            test_indices = indices[n_fold*i:]
            train_indices = indices[:n_fold*i]
        else:
            test_indices = indices[n_fold*i:n_fold*(i+1)]
            train_indices = np.concatenate([indices[:n_fold*i], indices[n_fold*(i+1):]])  
        result.append((train_indices, test_indices))
    return result

indices = my_KFold(X, y, 3, shuffle=False, random_state=21)
indices

[(array([16126, 16127, 16128, ..., 48376, 48377, 48378], shape=(32253,)),
  array([    0,     1,     2, ..., 16123, 16124, 16125], shape=(16126,))),
 (array([    0,     1,     2, ..., 48376, 48377, 48378], shape=(32253,)),
  array([16126, 16127, 16128, ..., 32249, 32250, 32251], shape=(16126,))),
 (array([    0,     1,     2, ..., 32249, 32250, 32251], shape=(32252,)),
  array([32252, 32253, 32254, ..., 48376, 48377, 48378], shape=(16127,)))]

Grouped K-Fold, where k and group_field are input parameters, returns list of train and test indices.

In [16]:
def grouped_KFold(X, y, k, group_field, shuffle=True, random_state=None):
    
    group_id = ''
    for i, group in enumerate(group_field):
        if i == 0:
            group_id += X[group].astype(str)
        else:
            group_id += '_' + X[group].astype(str)
    
    groups = group_id.values  
    unique_groups = np.unique(groups)
    
    if shuffle==True:
        if random_state != None:
            np.random.seed(random_state)
        np.random.shuffle(unique_groups)
        
    group_folds = np.array_split(unique_groups, k)
    result = []
    
    for i in range(k):
        test_groups = group_folds[i]
        train_groups = np.concatenate([group_folds[j] for j in range(k) if j != i])
        
        test_mask = np.isin(groups, test_groups)
        train_mask = np.isin(groups, train_groups)
        
        test_indices = np.where(test_mask)[0]
        train_indices = np.where(train_mask)[0]
        
        result.append((train_indices, test_indices))
        
    return result

indices = grouped_KFold(X, y, 5, ['bathrooms'], shuffle=True, random_state=21)
indices

[(array([    2,    16,    23, ..., 48351, 48357, 48370], shape=(8805,)),
  array([    0,     1,     3, ..., 48376, 48377, 48378], shape=(39574,))),
 (array([    0,     1,     2, ..., 48376, 48377, 48378], shape=(48146,)),
  array([  474,   782,   922,  2191,  2330,  2358,  2381,  2611,  3235,
          3834,  3891,  4307,  4358,  4963,  5094,  5383,  5456,  5570,
          5625,  5631,  5641,  5708,  5784,  5819,  6025,  6273,  6292,
          6393,  7014,  7106,  7613,  7643,  7759,  8142,  8221,  8811,
          8851,  8870,  9143,  9560,  9722,  9932,  9981, 10153, 10228,
         10510, 10516, 10584, 11012, 11084, 11268, 11764, 12200, 12205,
         12217, 12224, 12241, 12601, 12756, 13175, 13240, 13351, 14216,
         14554, 14640, 14775, 14872, 15244, 15327, 15330, 15722, 15994,
         16044, 16378, 16878, 16951, 17059, 17109, 17347, 17429, 17537,
         17626, 17852, 17980, 18697, 18846, 18888, 18938, 19082, 19238,
         19291, 19410, 19606, 19918, 19950, 20133, 20266, 

Stratified K-fold, where k and stratify_field are input parameters, returns list of train and test indices.

In [17]:
def stratified_Kfold(X, y, k, stratify_field, random_state=None, shuffle=True):
    
    if stratify_field == 'target':
        strata = y.values  
    elif stratify_field == 'price_strata':
        strata = df['price_strata'].values 
    else:
        strata = X[stratify_field].values  
        
    n_samples = len(X)
    strata_indices = {}
    
    for class_value in np.unique(strata):
        mask = (strata == class_value)
        strata_indices[class_value] = np.where(mask)[0] 
        
    strata_folds = {}
    for class_value, indices in strata_indices.items():
        if shuffle==True:
            if random_state != None:
                np.random.seed(random_state)
            np.random.shuffle(indices)
        strata_folds[class_value] = np.array_split(indices, k)
        
    result = []
    for fold_i in range(k):
        test_indices = []
        
        for class_value in strata_indices.keys():
            test_indices.extend(strata_folds[class_value][fold_i])

        all_indices = np.arange(n_samples)
        train_indices = np.setdiff1d(all_indices, test_indices)
        
        result.append((train_indices, np.array(test_indices)))
    
    return result
        
indices = stratified_Kfold(X, y, 5, stratify_field='bathrooms')
indices

[(array([    0,     1,     2, ..., 48375, 48376, 48377], shape=(38698,)),
  array([28732, 43651, 40928, ..., 15263,  8967,  4358], shape=(9681,))),
 (array([    1,     2,     3, ..., 48376, 48377, 48378], shape=(38701,)),
  array([41995, 41129, 10198, ..., 30067, 44557, 24042], shape=(9678,))),
 (array([    0,     1,     3, ..., 48375, 48376, 48378], shape=(38704,)),
  array([19465, 13875, 13881, ..., 13838, 24179, 32301], shape=(9675,))),
 (array([    0,     1,     2, ..., 48375, 48377, 48378], shape=(38706,)),
  array([47778, 43290, 11066, ..., 21447, 19534, 33081], shape=(9673,))),
 (array([    0,     2,     3, ..., 48376, 48377, 48378], shape=(38707,)),
  array([ 3686, 28665,  2061, ..., 27036, 35847, 37626], shape=(9672,)))]

Time series split, where k and date_field are input parameters, returns list of train and test indices.

In [18]:
def time_KFold(X, y, k, date_field):
    X_copy = X.copy()
    X_copy[date_field] = pd.to_datetime(X_copy[date_field])
    X_sorted = X_copy.sort_values(date_field)
    
    n_samples = len(X_sorted)
    result = []

    fold_size = n_samples // (k + 1)
    
    for i in range(1, k + 1):
        train_end = i * fold_size
        train_indices = np.arange(0, train_end)  
        
        if i == k:
            test_indices = np.arange(train_end, n_samples) 
        else:
            test_end = (i + 1) * fold_size
            test_indices = np.arange(train_end, test_end) 
        
        result.append((train_indices, test_indices))
    
    return result

X['created'] = df['created']

indices = time_KFold(X, y, 5, 'created')
indices

[(array([   0,    1,    2, ..., 8060, 8061, 8062], shape=(8063,)),
  array([ 8063,  8064,  8065, ..., 16123, 16124, 16125], shape=(8063,))),
 (array([    0,     1,     2, ..., 16123, 16124, 16125], shape=(16126,)),
  array([16126, 16127, 16128, ..., 24186, 24187, 24188], shape=(8063,))),
 (array([    0,     1,     2, ..., 24186, 24187, 24188], shape=(24189,)),
  array([24189, 24190, 24191, ..., 32249, 32250, 32251], shape=(8063,))),
 (array([    0,     1,     2, ..., 32249, 32250, 32251], shape=(32252,)),
  array([32252, 32253, 32254, ..., 40312, 40313, 40314], shape=(8063,))),
 (array([    0,     1,     2, ..., 40312, 40313, 40314], shape=(40315,)),
  array([40315, 40316, 40317, ..., 48376, 48377, 48378], shape=(8064,)))]

### Cross-validation comparison

In [19]:
df['price_strata'] = pd.qcut(df['price'], q=5, labels=False)

In [20]:
custom_kfold = my_KFold(X, y, k=5, shuffle=True, random_state=42)
custom_grouped = grouped_KFold(X, y, k=5, group_field=['bathrooms', 'bedrooms'], random_state=42)
custom_stratified = stratified_Kfold(X, y, k=5, stratify_field='price_strata', random_state=42)
custom_time = time_KFold(X, y, k=5, date_field='created')

sklearn_kfold = list(KFold(n_splits=5, shuffle=True, random_state=42).split(X, y))
sklearn_stratified = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, df['price_strata']))
sklearn_time = list(TimeSeriesSplit(n_splits=5).split(X, y))

groups = X['bathrooms'].astype(str) + '_' + X['bedrooms'].astype(str)
sklearn_grouped = list(GroupKFold(n_splits=5).split(X, y, groups=groups))

In [21]:
def final_stats_comparison():
    stats_data = []
    
    all_methods_final = {
        'Custom_KFold': custom_kfold,
        'Sklearn_KFold': sklearn_kfold,
        'Custom_Grouped': custom_grouped, 
        'Sklearn_Grouped': sklearn_grouped,
        'Custom_Stratified': custom_stratified,
        'Sklearn_Stratified': sklearn_stratified,
        'Custom_Time_Corrected': custom_time,
        'Sklearn_Time': sklearn_time
    }
    
    for method_name, splits in all_methods_final.items():
        method_diffs = []
        
        for fold, (train_idx, test_idx) in enumerate(splits):
            try:
                y_train = y.iloc[train_idx]
                y_test = y.iloc[test_idx]
                mean_diff = abs(y_train.mean() - y_test.mean())
                method_diffs.append(mean_diff)
            except:
                continue
        
        if method_diffs:
            stats_data.append({
                'Method': method_name,
                'Avg_Mean_Diff': np.mean(method_diffs),
                'Std_Mean_Diff': np.std(method_diffs),
                'Min_Mean_Diff': np.min(method_diffs),
                'Max_Mean_Diff': np.max(method_diffs),
                'Status': 'WORKING'
            })
        else:
            stats_data.append({
                'Method': method_name,
                'Avg_Mean_Diff': None,
                'Std_Mean_Diff': None,
                'Min_Mean_Diff': None,
                'Max_Mean_Diff': None,
                'Status': 'ERROR'
            })
    
    stats_df = pd.DataFrame(stats_data)
    print("FINAL COMPARISON RESULTS:")
    print(stats_df.sort_values('Avg_Mean_Diff').to_string(index=False))
    
    working_methods = stats_df[stats_df['Status'] == 'WORKING']
    if len(working_methods) > 0:
        best_method = working_methods.loc[working_methods['Avg_Mean_Diff'].idxmin()]
        print(f"\nBEST METHOD: {best_method['Method']}")
        print(f"   Average Mean Difference: {best_method['Avg_Mean_Diff']:.2f}")

final_stats_comparison()

FINAL COMPARISON RESULTS:
               Method  Avg_Mean_Diff  Std_Mean_Diff  Min_Mean_Diff  Max_Mean_Diff  Status
    Custom_Stratified       3.856640       2.117121       1.795437       7.846667 WORKING
   Sklearn_Stratified       4.198945       2.293157       2.073457       8.155074 WORKING
        Sklearn_KFold      11.442989       6.654247       0.383622      19.883904 WORKING
         Custom_KFold      11.567533       6.426475       0.715217      19.466896 WORKING
Custom_Time_Corrected      39.745544      36.240030       0.603932      91.972798 WORKING
         Sklearn_Time      39.843663      36.259325       0.621314      91.911515 WORKING
       Custom_Grouped    1175.864372     788.835577     268.075555    2208.067932 WORKING
      Sklearn_Grouped    1211.081429     597.529316     434.014986    2152.336654 WORKING

BEST METHOD: Custom_Stratified
   Average Mean Difference: 3.86


### Feature Selection

In [22]:
def clean_results():
    columns = ['model', 'train', 'valid', 'test']
    result_MAE = pd.DataFrame(columns=columns)
    result_RMSE = pd.DataFrame(columns=columns)
    result_R2 = pd.DataFrame(columns=columns)
    return result_MAE, result_RMSE, result_R2

result_MAE, result_RMSE, result_R2 = clean_results()

def write_results(model, pred_train, pred_valid, pred_test, y_train, y_valid, y_test):
    mae_train = mean_absolute_error(y_train, pred_train)
    mae_test = mean_absolute_error(y_test, pred_test)
    mae_valid = mean_absolute_error(y_valid, pred_valid)

    rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
    rmse_valid = np.sqrt(mean_squared_error(y_valid, pred_valid))
    
    r2_train = r2_score(y_train, pred_train)
    r2_test = r2_score(y_test, pred_test)
    r2_valid = r2_score(y_valid, pred_valid)

    result_MAE.loc[len(result_MAE)] = [model, mae_train, mae_valid, mae_test]
    result_RMSE.loc[len(result_RMSE)] = [model, rmse_train, rmse_valid, rmse_test]
    result_R2.loc[len(result_R2)] = [model, r2_train, r2_valid, r2_test]

In [23]:
X = X.drop('created', axis=1)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, test_size=0.2, val_size=0.2, random_state=21)

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

lasso = Lasso(alpha=1, random_state=21)
lasso.fit(X_train_scaled, y_train)
    
y_train_pred = lasso.predict(X_train_scaled)
y_val_pred = lasso.predict(X_val_scaled)
y_test_pred = lasso.predict(X_test_scaled)

write_results('lasso', y_train_pred, y_val_pred, y_test_pred, y_train, y_val, y_test)

In [24]:
feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': np.abs(lasso.coef_)
    }).sort_values('importance', ascending=False)

print("Top 10 most important features:")
print(feature_importance.head(10))

top_features = list(feature_importance['feature'])

Top 10 most important features:
              feature    importance
0           bathrooms  14675.285414
1            bedrooms   3843.986449
2      interest_level    843.046872
7             Doorman    566.149872
13      LaundryinUnit    420.495515
3            Elevator    212.143346
20  LaundryInBuilding    201.013672
17  HighSpeedInternet    185.612632
11      FitnessCenter    180.417858
10  LaundryinBuilding    180.326861


In [25]:
X_train_top = X_train[top_features]
X_val_top = X_val[top_features]
X_test_top = X_test[top_features]

X_train_scaled = scaler.fit_transform(X_train_top)
X_val_scaled = scaler.transform(X_val_top)
X_test_scaled = scaler.transform(X_test_top)

lasso.fit(X_train_scaled, y_train)
    
y_train_pred = lasso.predict(X_train_scaled)
y_val_pred = lasso.predict(X_val_scaled)
y_test_pred = lasso.predict(X_test_scaled)

write_results('lasso_top10', y_train_pred, y_val_pred, y_test_pred, y_train, y_val, y_test)

In [26]:
def permutation_importance_normalized(model, X, y, n_repeats=10, random_state=42):

    if random_state is not None:
        np.random.seed(random_state)

    baseline_score = mean_squared_error(y, model.predict(X))
    
    importance_scores = {feature: [] for feature in X.columns}
    
    for feature in X.columns:
        X_permuted = X.copy()
        
        for _ in range(n_repeats):
            X_permuted[feature] = np.random.permutation(X_permuted[feature])
            
            y_pred_permuted = model.predict(X_permuted)
            permuted_score = mean_squared_error(y, y_pred_permuted)
            
            importance = (permuted_score - baseline_score) / baseline_score
            importance_scores[feature].append(importance)
            
            X_permuted[feature] = X[feature]
    
    importance_df = pd.DataFrame({
        'feature': list(importance_scores.keys()),
        'importance_mean': [np.mean(scores) for scores in importance_scores.values()],
        'importance_std': [np.std(scores) for scores in importance_scores.values()],
        'importance_scores': list(importance_scores.values())
    }).sort_values('importance_mean', ascending=False)
    
    return importance_df

def select_top_features_permutation(model, X, y, n_features=10, n_repeats=10, random_state=42):
    
    perm_importance = permutation_importance_normalized(model, X, y, n_repeats=n_repeats, random_state=random_state)
    
    top_features = perm_importance.head(n_features)['feature'].values
    
    print(f"Selected top {n_features} features by Normalized Permutation Importance:")
    for i, feature in enumerate(top_features, 1):
        importance = perm_importance[perm_importance['feature'] == feature]['importance_mean'].values[0]
        std = perm_importance[perm_importance['feature'] == feature]['importance_std'].values[0]
        print(f"{i:2d}. {feature:25} (importance: {importance:.4f} ± {std:.4f})")
    
    return top_features, perm_importance

In [27]:
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

lasso.fit(X_train_scaled, y_train)

top_10_perm, perm_importance_df = select_top_features_permutation(
    lasso, 
    pd.DataFrame(X_val_scaled, columns=X_train.columns), 
    y_val,
    n_features=10, 
    n_repeats=5, 
    random_state=21
)

Selected top 10 features by Normalized Permutation Importance:
 1. bathrooms                 (importance: 0.9008 ± 0.0206)
 2. bedrooms                  (importance: 0.5492 ± 0.0083)
 3. Doorman                   (importance: 0.1639 ± 0.0058)
 4. interest_level            (importance: 0.1328 ± 0.0078)
 5. LaundryinUnit             (importance: 0.0542 ± 0.0023)
 6. Elevator                  (importance: 0.0245 ± 0.0009)
 7. FitnessCenter             (importance: 0.0171 ± 0.0008)
 8. LaundryinBuilding         (importance: 0.0147 ± 0.0015)
 9. Dishwasher                (importance: 0.0092 ± 0.0012)
10. HardwoodFloors            (importance: 0.0072 ± 0.0011)


In [28]:
X_train_top = X_train[top_10_perm]
X_val_top = X_val[top_10_perm]
X_test_top = X_test[top_10_perm]

X_train_scaled = scaler.fit_transform(X_train_top)
X_val_scaled = scaler.transform(X_val_top)
X_test_scaled = scaler.transform(X_test_top)

lasso.fit(X_train_scaled, y_train)
    
y_train_pred = lasso.predict(X_train_scaled)
y_val_pred = lasso.predict(X_val_scaled)
y_test_pred = lasso.predict(X_test_scaled)

write_results('lasso_perm', y_train_pred, y_val_pred, y_test_pred, y_train, y_val, y_test)

In [29]:
def get_top_10_shap(X, y, alpha=1.0, sample_size=500):

    X_scaled = scaler.fit_transform(X)
    
    lasso.fit(X_scaled, y)

    if len(X_scaled) > sample_size:
        X_sample = X_scaled[:sample_size]
    else:
        X_sample = X_scaled
    
    explainer = shap.LinearExplainer(lasso, X_sample, feature_perturbation="interventional")
    shap_values = explainer.shap_values(X_sample)
    
    shap_importance = np.abs(shap_values).mean(0)
    
    top_10_idx = np.argsort(shap_importance)[-10:][::-1]
    top_10_features = X.columns[top_10_idx].values
    
    print("Top 10 SHAP features:")
    for i, feature in enumerate(top_10_features, 1):
        idx = list(X.columns).index(feature)
        print(f"{i}. {feature} (importance: {shap_importance[idx]:.4f})")
    
    return top_10_features

top_10_shap = get_top_10_shap(X_train, y_train, alpha=1)

Top 10 SHAP features:
1. bathrooms (importance: 485.7519)
2. bedrooms (importance: 456.6656)
3. Doorman (importance: 275.1035)
4. interest_level (importance: 240.7067)
5. LaundryinUnit (importance: 127.3429)
6. Elevator (importance: 105.5286)
7. LaundryinBuilding (importance: 80.3176)
8. FitnessCenter (importance: 71.5682)
9. Dishwasher (importance: 68.9918)
10. HardwoodFloors (importance: 55.8315)


In [30]:
X_train_top = X_train[top_10_shap]
X_val_top = X_val[top_10_shap]
X_test_top = X_test[top_10_shap]

X_train_scaled = scaler.fit_transform(X_train_top)
X_val_scaled = scaler.transform(X_val_top)
X_test_scaled = scaler.transform(X_test_top)

lasso.fit(X_train_scaled, y_train)
    
y_train_pred = lasso.predict(X_train_scaled)
y_val_pred = lasso.predict(X_val_scaled)
y_test_pred = lasso.predict(X_test_scaled)

write_results('lasso_shap', y_train_pred, y_val_pred, y_test_pred, y_train, y_val, y_test)

In [31]:
result_MAE

,model,train,valid,test
0,lasso,683.884224,694.995757,698.159939
1,lasso_top10,683.884428,694.995814,698.160376
2,lasso_perm,687.568268,698.141919,702.865501
3,lasso_shap,687.568852,698.142109,702.865877


In [32]:
result_RMSE

,model,train,valid,test
0,lasso,997.508886,1015.627070,1016.884695
1,lasso_top10,997.508880,1015.626965,1016.884940
2,lasso_perm,1003.167211,1022.192854,1023.267419
3,lasso_shap,1003.167210,1022.192479,1023.267396


In [33]:
result_R2

,model,train,valid,test
0,lasso,0.604320,0.602122,0.606125
1,lasso_top10,0.604320,0.602122,0.606125
2,lasso_perm,0.599818,0.596961,0.601165
3,lasso_shap,0.599818,0.596961,0.601165


### Hyperparameter optimization

In [34]:
def elasticnet_grid_search(X_train, X_val, y_train, y_val, top_features=None):

    X_train_search = X_train[top_features]
    X_val_search = X_val[top_features]
    
    X_train_scaled = scaler.fit_transform(X_train_search)
    X_val_scaled = scaler.transform(X_val_search)

    param_grid = {
        'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]  
    }
    
    elastic_net = ElasticNet(random_state=42, max_iter=10000)

    grid_search = GridSearchCV(
        elastic_net, 
        param_grid, 
        cv=3, 
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train_scaled, y_train)

    best_params = grid_search.best_params_
    best_score = -grid_search.best_score_
    
    print(f"\nGrid Search Results:")
    print(f"Best parameters: {best_params}")
    print(f"Best CV MSE: {best_score:.4f}")
 
    best_model = grid_search.best_estimator_
    y_val_pred = best_model.predict(X_val_scaled)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)
    
    print(f"Validation RMSE: {val_rmse:.4f}")
    print(f"Validation R²: {val_r2:.4f}")
    
    results_df = pd.DataFrame(grid_search.cv_results_)
    results_df = results_df[['param_alpha', 'param_l1_ratio', 'mean_test_score', 'std_test_score']]
    results_df['mean_test_score'] = -results_df['mean_test_score'] 
    
    print(f"\nTop 5 parameter combinations:")
    print(results_df.sort_values('mean_test_score').head(5).to_string(index=False))
    
    return best_model, scaler, best_params, results_df

best_elasticnet_grid, scaler_grid, best_params_grid, grid_results = elasticnet_grid_search(
    X_train, X_val, y_train, y_val, top_features=top_features
)

Fitting 3 folds for each of 36 candidates, totalling 108 fits

Grid Search Results:
Best parameters: {'alpha': 0.01, 'l1_ratio': 1.0}
Best CV MSE: 997287.0629
Validation RMSE: 1015.3796
Validation R²: 0.6023

Top 5 parameter combinations:
 param_alpha  param_l1_ratio  mean_test_score  std_test_score
       0.010             1.0    997287.062945    11283.505080
       0.001             1.0    997287.711528    11270.559352
       0.100             1.0    997288.586300    11413.186287
       1.000             1.0    997965.183528    12780.647179
       0.001             0.9    998551.326171    14399.782291


In [35]:
optuna.logging.set_verbosity(optuna.logging.WARNING) 

def objective_elasticnet(trial, X_train, y_train, X_val, y_val, top_features=None, kfold=True, n_splits=5):

    alpha = trial.suggest_categorical('alpha', [0.001, 0.01, 0.1, 1.0, 10.0, 100.0])
    l1_ratio = trial.suggest_categorical('l1_ratio', [0.1, 0.3, 0.5, 0.7, 0.9, 1.0])
    
    if top_features is not None:
        X_train_opt = X_train[top_features]
        X_val_opt = X_val[top_features]
    else:
        X_train_opt = X_train
        X_val_opt = X_val

    try:
        if kfold:
            from sklearn.model_selection import KFold
            kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
            
            cv_scores = []
            
            for train_idx, val_idx in kf.split(X_train_opt):
                X_train_fold = X_train_opt.iloc[train_idx]
                X_val_fold = X_train_opt.iloc[val_idx]
                y_train_fold = y_train.iloc[train_idx]
                y_val_fold = y_train.iloc[val_idx]
                
                X_train_scaled = scaler.fit_transform(X_train_fold)
                X_val_scaled = scaler.transform(X_val_fold)
                
                model = ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1_ratio,
                    random_state=42,
                    max_iter=10000
                )
                
                model.fit(X_train_scaled, y_train_fold)
                y_val_pred = model.predict(X_val_scaled)
                mse = mean_squared_error(y_val_fold, y_val_pred)
                cv_scores.append(mse)
            
            mse = np.mean(cv_scores)
            
            trial.set_user_attr('std_cv_mse', np.std(cv_scores))
            trial.set_user_attr('cv_scores', cv_scores)
            
        else:
            X_train_scaled = scaler.fit_transform(X_train_opt)
            X_val_scaled = scaler.transform(X_val_opt)

            model = ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                random_state=42,
                max_iter=10000
            )
            
            model.fit(X_train_scaled, y_train)
            y_val_pred = model.predict(X_val_scaled)
            mse = mean_squared_error(y_val, y_val_pred)
            
            trial.set_user_attr('std_cv_mse', 0.0)  
            trial.set_user_attr('cv_scores', [mse])
        
        r2 = 1 - mse / np.var(y_val if not kfold else y_train)
        
        X_full_scaled = scaler.fit_transform(X_train_opt)
        final_model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            random_state=42,
            max_iter=10000
        )
        final_model.fit(X_full_scaled, y_train)
        
        trial.set_user_attr('r2', r2)
        trial.set_user_attr('non_zero_coef', np.sum(final_model.coef_ != 0))
        trial.set_user_attr('method', 'kfold' if kfold else 'split')
        
        return mse
        
    except Exception as e:
        return float('inf')

In [36]:
study = optuna.create_study(
    direction='minimize',  # минимизируем MSE
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    lambda trial: objective_elasticnet(
        trial, 
        X_train,  
        y_train, 
        X_val,    
        y_val,
        top_features=top_features, 
        kfold = False  
    ), 
    n_trials=20  
)

print("Best parameters:", study.best_params)
print("Best MSE:", study.best_value)

Best parameters: {'alpha': 0.01, 'l1_ratio': 1.0}
Best MSE: 1030995.6353181785


In [37]:
study.optimize(
    lambda trial: objective_elasticnet(
        trial, 
        X_train, y_train, X_val, y_val,
        top_features=top_features,
        kfold=True,      # используем K-Fold
        n_splits=5       # 5 фолдов
    ),
    n_trials=20
)

print("Best parameters:", study.best_params)
print("Best MSE:", study.best_value)

Best parameters: {'alpha': 0.01, 'l1_ratio': 1.0}
Best MSE: 996408.603279545
